# DeMoSearch · single-photo and multiview wall analysis
2D: one photo → SAM3 masks / YOLO26 type proposals → scene.json.
3D: photos/video → COLMAP mesh → masks / multiview correspondence → the same scene.json schema.
All implementation lives in `src/hold_detection`; this notebook only calls the CLI.
The route algorithm consumes the exported holds. No beta is generated here.

In [ ]:
# @title Set up the environment (once)
import json, os, subprocess, sys, uuid
from pathlib import Path
from datetime import datetime, timezone

CODE_REF = 'main'  # @param {type: "string"}
WORK = Path('/content/demosearch-wall')
PROJECT = WORK / 'project'
WORK.mkdir(parents=True, exist_ok=True)
repository = 'https://github.com/DiasKzStrong/hold_detection.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--quiet', '--depth', '1', repository, str(PROJECT)], check=True)
if not (PROJECT / '.git').is_dir():
    raise RuntimeError('The project directory contains unrelated files; choose another WORK path.')
dirty = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
if dirty.strip():
    raise RuntimeError('Save local project changes before updating; they will not be overwritten.')
subprocess.run(['git', '-C', str(PROJECT), 'fetch', '--quiet', 'origin', CODE_REF], check=True)
CODE_SHA = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
previous = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if previous != CODE_SHA and 'hold_detection' in sys.modules:
    raise RuntimeError('Source code has changed: restart the runtime to avoid mixing module versions.')
subprocess.run(['git', '-C', str(PROJECT), 'checkout', '--quiet', '--detach', CODE_SHA], check=True)
if str(PROJECT / 'src') not in sys.path: sys.path.insert(0, str(PROJECT / 'src'))
from hold_detection.notebook_runtime import prepare_runtime, run_cli, runtime_info
PYTHON = prepare_runtime(WORK, PROJECT)
MODEL_ENV = runtime_info(PYTHON, PROJECT)

def run_holds(*arguments, log_name='command'):
    return run_cli(PYTHON, PROJECT, list(arguments), WORK / 'logs' / (log_name + '.log'))

print('Ready ·', MODEL_ENV['gpu'] if MODEL_ENV['cuda_available'] else 'CPU baseline')
print('Kernel Python:', sys.version.split()[0], '| model Python:', MODEL_ENV['python'])
print('Code commit:', CODE_SHA, '| installation logs:', WORK / 'logs')

In [ ]:
import getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('Approved Hugging Face read token (hidden): ')
PYTHON = prepare_runtime(WORK, PROJECT, sam3=True)
run_holds('fetch-sam3', log_name='fetch_sam3')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MODE = '2d'  # @param ["2d", "3d"]
SOURCE = Path('/content/drive/MyDrive/DeMoSearch/wall.jpg')
TYPE_WEIGHTS = ''  # Optional fine-tuned six-class YOLO26 best.pt; empty means review types manually.
OUTPUT = Path('/content/drive/MyDrive/DeMoSearch/analysis_demo')
COLMAP = 'colmap'  # 3D requires a separately installed CUDA COLMAP executable/container.
SCALE_M_PER_UNIT = None  # Leave None if no scale calibration is available.

In [ ]:
args = ['analyze-wall', SOURCE, '--mode', MODE, '--output', OUTPUT,
        '--sam-weights', PROJECT / 'models/sam3/sam3.pt', '--device', '0', '--colmap', COLMAP]
if TYPE_WEIGHTS: args += ['--type-weights', TYPE_WEIGHTS]
if SCALE_M_PER_UNIT is not None: args += ['--scale-m-per-unit', SCALE_M_PER_UNIT]
run_holds(*args, log_name='wall_analysis')

In [ ]:
import pandas as pd
from IPython.display import display
scene = json.loads((OUTPUT / 'scene.json').read_text())
print('Mode:', scene['analysis_mode'], '| holds:', len(scene['holds']))
print('Units:', scene['wall']['units'], '| review:', scene['review'])
display(pd.DataFrame(scene['holds']).head(30))
print('Graph algorithm input:', OUTPUT / 'scene.json')
print('Reconstructed GLB:', scene['wall']['glb_path'])

For a full experiment use `scripts/slurm/submit_wall.sh`. It schedules preparation,
reconstruction (3D only), segmentation and JSON export with afterok dependencies.
See `docs/wall-pipeline.md`. No HPC job is submitted by opening this notebook.